# Microbial Isolate RNA Sequence Features and Annotations from Oxford Nanopore and Second-Generation Sequencing Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.6vvr-dn1t/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print("Dataset Name: {}".format(metadata.name))
print("Description: {}".format(metadata.description))
print("Version: {}".format(metadata.version))
print("License: {}".format(metadata.license))

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will inspect all the available record sets, their fields, and columns for the dataset. Every entity is referenced by its `@id`.

In [ ]:
# Get all record sets from metadata
record_sets = metadata.recordSet
if not record_sets:
    print("No record sets found in this dataset. If your version of mlcroissant supports `.record_sets`, you may use that instead.")
else:
    for rs in record_sets:
        print("RecordSet @id:", rs['@id'])
        print("  Name:", rs.get('name', None))
        fields = rs.get('field', [])
        if fields:
            for field in fields:
                print("    Field @id:", field['@id'])
                print("      Name:", field.get('name', None))
                print("      DataType:", field.get('dataType', None))
                print("      Column(s):")
                columns = field.get('column', [])
                if columns:
                    for col in columns:
                        print("        Column @id:", col['@id'])
                        print("          Name:", col.get('name', None))
                        print("          DataType:", col.get('dataType', None))
                else:
                    print("        (No columns listed)")
        else:
            print("    (No fields listed)")
        print('---')

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

Note: This section follows the Croissant schema structure and references all dataset entities by their `@id`. Modify the values below to match the record set IDs from the overview.

In [ ]:
# Prepare to extract all available record sets
# If empty, skip extraction.
record_set_ids = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    record_set_ids = [rs['@id'] for rs in metadata.recordSet]

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\n[RecordSet @id] {record_set_id}: ")
    print(df.columns.tolist())
    print(df.head(3))

# Example: Use the first record set for further processing
if record_set_ids:
    focus_recordset_id = record_set_ids[0]
else:
    focus_recordset_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
This section includes operations like removing outliers, transforming distributions, or grouping by key attributes to prepare for further analysis.

**Remember:** All fields must be referenced by their `@id`, as listed in the previous overview.

In [ ]:
import numpy as np
import warnings
warnings.simplefilter('ignore', FutureWarning)

# Attempt EDA on the focus_recordset
if focus_recordset_id and focus_recordset_id in dataframes:
    df = dataframes[focus_recordset_id]
    print("Working with RecordSet @id:", focus_recordset_id)
    # Infer numeric fields (for demonstration, select a column named 'length', or guess numeric columns)
    numeric_fields = [col for col in df.columns if df[col].dtype in [np.int64, np.float64]]
    if not numeric_fields:
        # Try fallback: look for typical field names (replace as needed for your schema)
        possible_numeric = [col for col in df.columns if 'length' in col or 'count' in col or 'score' in col or 'coverage' in col or 'start' in col or 'end' in col]
        numeric_fields = possible_numeric

    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Selected numeric field (by @id): {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if not df[numeric_field_id].empty else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical field. Select, e.g., 'sample_id' or similar as group field.
        group_fields = [col for col in df.columns if df[col].dtype == object and ('sample' in col or 'type' in col or 'strand' in col or 'organism' in col)]
        group_field = group_fields[0] if group_fields else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of {numeric_field_id} by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric field detected for EDA. Modify code to select available fields by their @id.")
else:
    print("No record set DataFrame is available for EDA. Check previous extraction.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All fields and attributes must be referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Example: Visualize the distribution of the selected numeric field
if focus_recordset_id and focus_recordset_id in dataframes and 'numeric_field_id' in locals():
    df = dataframes[focus_recordset_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (RecordSet @id: {focus_recordset_id})")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
    # Optionally, visualize group_field relationships if available
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.show()
else:
    print("No numeric field or record set available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The Croissant schema provides structured access to multi-modal sequencing and annotation data for microbial isolates.
- Record sets, fields, and columns were programmatically referenced by their `@id` values as required.
- Exploratory steps enabled basic field selection, normalization, and grouping.
- This analysis framework can be extended for deeper bioinformatics, comparative genomics, or feature annotation analyses using the FAIR^2 dataset package and Croissant's metadata mapping.